# Instruction (Read this)
- Use this template to develop your project. Do not change the steps. 
- For each step, you may add additional cells if needed.
- But remove <b>unnecessary</b> cells to ensure the notebook is readable.
- Marks will be <b>deducted</b> if the notebook is cluttered or difficult to follow due to excess or irrelevant content.
- <b>Briefly</b> describe the steps in the "Description:" field.
- <b>Do not</b> submit the dataset. 
- The submitted jupyter notebook will be executed using the uploaded dataset in eLearn.

# Group Information

Group No:

- Member 1: CH'NG BAO SHENG (23302782)
- Member 2: LIM YONG ZHOU (23302902)
- Member 3: JEFFRY SOO YU ZHEN (23304759)

# Import Necessary Libraries

## Core Data Handling

In [ ]:
import pandas as pd # for data manipulation

## Numerics and Array Operations

In [ ]:
import numpy as np # for numerical operations

## Machine Learning Algorithms and Utilities

In [ ]:
from sklearn.linear_model import LogisticRegression # for Logistic Regression classifier
from sklearn.svm import SVC # for Support Vector Machine classifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score # for splitting data and cross-validation
from sklearn.preprocessing import StandardScaler # for feature scaling
from sklearn.feature_selection import SequentialFeatureSelector # for Sequential Feature Selection

## Visualization

In [ ]:
import matplotlib.pyplot as plt # for plotting graphs
import seaborn as sns # for enhanced data visualization

# Load the Dataset

In [ ]:
# Define column names based on the Dermatology dataset README
columns = [
    'erythema',
    'scaling',
    'definite_borders',
    'itching',
    'koebner_phenomenon',
    'polygonal_papules',
    'follicular_papules',
    'oral_mucosal_involvement',
    'knee_and_elbow_involvement',
    'scalp_involvement',
    'family_history',
    'melanin_incontinence',
    'eosinophils_in_the_infiltrate',
    'PNL_infiltrate',
    'fibrosis_of_the_papillary_dermis',
    'exocytosis',
    'acanthosis',
    'hyperkeratosis',
    'parakeratosis',
    'clubbing_of_the_rete_ridges',
    'elongation_of_the_rete_ridges',
    'thinning_of_the_suprapapillary_epidermis',
    'spongiform_pustule',
    'munro_microabcess',
    'focal_hypergranulosis',
    'disappearance_of_the_granular_layer',
    'vacuolisation_and_damage_of_basal_layer',
    'spongiosis',
    'saw_tooth_appearance_of_retes',
    'follicular_horn_plug',
    'perifollicular_parakeratosis',
    'inflammatory_mononuclear_infiltrate',
    'band_like_infiltrate',
    'age',
    'class'
]

# Load the dataset without headers and convert '?' missing values to NaN
df = pd.read_csv(
    'Project Datasets/Dermatology/dermatology.csv',
    header=None,
    names=columns,
    na_values='?'
)

# Display the first five rows of the dataset
df.head()

# Exploratory Data Analysis (EDA)

## Initial Analysis of Dataset

In [ ]:
print('Dataset dimension:', df.shape) # Display the number of rows and columns
print('Number of features:', df.drop(columns=['class']).shape[1]) # Display the number of input features
print('Columns of features:', df.drop(columns=['class']).columns) # Display all feature column names
print('Target column: class') # Display the target column name

## Data Types of Dataset

In [ ]:
df.info() # Display column names, data types and non-null counts

## Descriptive Statistics for Numeric Columns

In [ ]:
df.describe() # Display descriptive statistics for numeric columns

# Data Preparation
______________________________________________________________________________________
Description: Missing values, duplicate rows, and outlier considerations are handled before checking the final class distribution and splitting the dataset.

## Data Cleaning

### Handling Missing Values

In [ ]:
missing_counts = df.isnull().sum() # Count missing values for each column
missing_counts[missing_counts > 0] # Display only columns with missing values

In [ ]:
age_median = df['age'].median() # Calculate the median age for imputation
df['age'] = df['age'].fillna(age_median) # Replace missing age values with the median age

print('Median age used for imputation:', age_median) # Display the median value used
print('Total missing values after imputation:', df.isnull().sum().sum()) # Check remaining missing values

### Handling Duplicate Rows

In [ ]:
duplicate_count = df.duplicated().sum() # Count duplicate rows in the dataset
print('Number of duplicate rows:', duplicate_count) # Display the number of duplicated rows

# Remove duplicate rows if any are found
if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True) # Drop duplicates and reset the index
    print('Duplicates removed. New dataset dimension:', df.shape) # Display new dataset dimension
else:
    print('No duplicate rows were found.') # Inform that no duplicates were found

## Outlier Checking
Most dermatology features are ordinal clinical or histopathological scores from 0 to 3. Therefore, values in this range are valid medical severity levels rather than statistical outliers. Only `age` is inspected separately, and rows are not removed using IQR to avoid deleting valid medical cases from a small dataset.

In [ ]:
plt.figure(figsize=(10, 4)) # Set the figure size for side-by-side plots

plt.subplot(1, 2, 1) # Create the first subplot for boxplot
sns.boxplot(y=df['age']) # Plot age values to inspect possible outliers
plt.title('Age Boxplot') # Set title for the boxplot

plt.subplot(1, 2, 2) # Create the second subplot for histogram
sns.histplot(df['age'], bins=20, kde=True) # Plot the distribution of age
plt.title('Age Distribution') # Set title for the histogram

plt.tight_layout() # Adjust layout to prevent overlapping
plt.show() # Display the plots

## Class Distribution After Cleaning

In [ ]:
# Define class labels based on the Dermatology dataset README
class_names = {
    1: 'psoriasis',
    2: 'seboreic dermatitis',
    3: 'lichen planus',
    4: 'pityriasis rosea',
    5: 'cronic dermatitis',
    6: 'pityriasis rubra pilaris'
}

# Count the number of instances for each disease class after cleaning
class_distribution = df['class'].value_counts().sort_index()

# Convert the class distribution into a DataFrame for clearer display
class_distribution_df = class_distribution.rename_axis('class').reset_index(name='count')

# Map each class number to its disease name
class_distribution_df['disease'] = class_distribution_df['class'].map(class_names)

# Display the final class distribution table before splitting
class_distribution_df

In [ ]:
plt.figure(figsize=(8, 5)) # Set the figure size
sns.countplot(data=df, x='class', order=sorted(df['class'].unique())) # Plot the number of records for each class
plt.title('Class Distribution of Dermatology Dataset After Cleaning') # Set the chart title
plt.xlabel('Disease Class') # Set x-axis label
plt.ylabel('Number of Instances') # Set y-axis label
plt.show() # Display the plot

# Split the Dataset
Split the dataset into training, validation, and test sets. Three stratified split ratios are prepared: 80/10/10, 70/15/15, and 60/20/20.

In [ ]:
X = df.drop(columns=['class']) # Select all input features
y = df['class'] # Select the multi-class target variable

print('All feature data types:') # Display the data types of all input features
print(X.dtypes.value_counts()) # Count the number of columns for each data type
print('\nTarget classes:', sorted(y.unique())) # Display unique target classes

In [ ]:
# Split the dataset into 80% training data and 20% temporary data for the 80/10/10 split
x_train_80, x_temp_20_for_80, y_train_80, y_temp_20_for_80 = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Split the temporary 20% data equally into 10% validation data and 10% test data
x_val_10, x_test_10, y_val_10, y_test_10 = train_test_split(
    x_temp_20_for_80,
    y_temp_20_for_80,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_20_for_80
)

# Split the dataset into 70% training data and 30% temporary data for the 70/15/15 split
x_train_70, x_temp_30_for_70, y_train_70, y_temp_30_for_70 = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split the temporary 30% data equally into 15% validation data and 15% test data
x_val_15, x_test_15, y_val_15, y_test_15 = train_test_split(
    x_temp_30_for_70,
    y_temp_30_for_70,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_30_for_70
)

# Split the dataset into 60% training data and 40% temporary data for the 60/20/20 split
x_train_60, x_temp_40_for_60, y_train_60, y_temp_40_for_60 = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

# Split the temporary 40% data equally into 20% validation data and 20% test data
x_val_20, x_test_20, y_val_20, y_test_20 = train_test_split(
    x_temp_40_for_60,
    y_temp_40_for_60,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_40_for_60
)

# Store all explicit split variables in a dictionary for easier looping in later preprocessing steps
split_datasets = {
    '80_10_10': {
        'X_train': x_train_80,
        'X_val': x_val_10,
        'X_test': x_test_10,
        'y_train': y_train_80,
        'y_val': y_val_10,
        'y_test': y_test_10
    },
    '70_15_15': {
        'X_train': x_train_70,
        'X_val': x_val_15,
        'X_test': x_test_15,
        'y_train': y_train_70,
        'y_val': y_val_15,
        'y_test': y_test_15
    },
    '60_20_20': {
        'X_train': x_train_60,
        'X_val': x_val_20,
        'X_test': x_test_20,
        'y_train': y_train_60,
        'y_val': y_val_20,
        'y_test': y_test_20
    }
}

# Display the dimensions of each explicit split dataset
print('80/10/10 split')
print('x_train_80:', x_train_80.shape, 'y_train_80:', y_train_80.shape)
print('x_val_10:', x_val_10.shape, 'y_val_10:', y_val_10.shape)
print('x_test_10:', x_test_10.shape, 'y_test_10:', y_test_10.shape)
print('-' * 50)

print('70/15/15 split')
print('x_train_70:', x_train_70.shape, 'y_train_70:', y_train_70.shape)
print('x_val_15:', x_val_15.shape, 'y_val_15:', y_val_15.shape)
print('x_test_15:', x_test_15.shape, 'y_test_15:', y_test_15.shape)
print('-' * 50)

print('60/20/20 split')
print('x_train_60:', x_train_60.shape, 'y_train_60:', y_train_60.shape)
print('x_val_20:', x_val_20.shape, 'y_val_20:', y_val_20.shape)
print('x_test_20:', x_test_20.shape, 'y_test_20:', y_test_20.shape)

In [ ]:
# Display the class distribution for each train, validation and test split
for split_name, data in split_datasets.items():
    split_distribution = pd.DataFrame({
        'train': data['y_train'].value_counts().sort_index(),
        'validation': data['y_val'].value_counts().sort_index(),
        'test': data['y_test'].value_counts().sort_index()
    })

    print(f'Class distribution for {split_name} split') # Display split name
    display(split_distribution) # Display the class distribution table for the split

In [ ]:
# Use the 70/15/15 split as the default split for the remaining preprocessing examples
X_train = x_train_70 # Default training features
X_val = x_val_15 # Default validation features
X_test = x_test_15 # Default test features
y_train = y_train_70 # Default training target
y_val = y_val_15 # Default validation target
y_test = y_test_15 # Default test target

# Data Preprocessing
Perform data preprocessing such as normalization, standardization, label encoding etc.
______________________________________________________________________________________
Description: Label encoding is not required because all attributes and class labels are already numeric. Standardization is applied after splitting. Each scaler is fitted only on its training set, then applied to the matching validation and test sets to avoid data leakage.

In [ ]:
# Create separate StandardScaler objects for each split ratio
scaler_80_10_10 = StandardScaler() # Scaler for 80/10/10 split
scaler_70_15_15 = StandardScaler() # Scaler for 70/15/15 split
scaler_60_20_20 = StandardScaler() # Scaler for 60/20/20 split

# Fit the scaler on 80% training data and transform the matching validation and test data
x_train_80_scaled = scaler_80_10_10.fit_transform(x_train_80) # Fit and transform 80% training features
x_val_10_scaled = scaler_80_10_10.transform(x_val_10) # Transform 10% validation features
x_test_10_scaled = scaler_80_10_10.transform(x_test_10) # Transform 10% test features

# Convert the 80/10/10 scaled arrays back into DataFrames
x_train_80_scaled_df = pd.DataFrame(x_train_80_scaled, columns=X.columns, index=x_train_80.index)
x_val_10_scaled_df = pd.DataFrame(x_val_10_scaled, columns=X.columns, index=x_val_10.index)
x_test_10_scaled_df = pd.DataFrame(x_test_10_scaled, columns=X.columns, index=x_test_10.index)

# Fit the scaler on 70% training data and transform the matching validation and test data
x_train_70_scaled = scaler_70_15_15.fit_transform(x_train_70) # Fit and transform 70% training features
x_val_15_scaled = scaler_70_15_15.transform(x_val_15) # Transform 15% validation features
x_test_15_scaled = scaler_70_15_15.transform(x_test_15) # Transform 15% test features

# Convert the 70/15/15 scaled arrays back into DataFrames
x_train_70_scaled_df = pd.DataFrame(x_train_70_scaled, columns=X.columns, index=x_train_70.index)
x_val_15_scaled_df = pd.DataFrame(x_val_15_scaled, columns=X.columns, index=x_val_15.index)
x_test_15_scaled_df = pd.DataFrame(x_test_15_scaled, columns=X.columns, index=x_test_15.index)

# Fit the scaler on 60% training data and transform the matching validation and test data
x_train_60_scaled = scaler_60_20_20.fit_transform(x_train_60) # Fit and transform 60% training features
x_val_20_scaled = scaler_60_20_20.transform(x_val_20) # Transform 20% validation features
x_test_20_scaled = scaler_60_20_20.transform(x_test_20) # Transform 20% test features

# Convert the 60/20/20 scaled arrays back into DataFrames
x_train_60_scaled_df = pd.DataFrame(x_train_60_scaled, columns=X.columns, index=x_train_60.index)
x_val_20_scaled_df = pd.DataFrame(x_val_20_scaled, columns=X.columns, index=x_val_20.index)
x_test_20_scaled_df = pd.DataFrame(x_test_20_scaled, columns=X.columns, index=x_test_20.index)

# Store all explicit scaled variables in a dictionary for easier looping in later model training
scaled_datasets = {
    '80_10_10': {
        'X_train_scaled': x_train_80_scaled_df,
        'X_val_scaled': x_val_10_scaled_df,
        'X_test_scaled': x_test_10_scaled_df,
        'y_train': y_train_80,
        'y_val': y_val_10,
        'y_test': y_test_10,
        'scaler': scaler_80_10_10
    },
    '70_15_15': {
        'X_train_scaled': x_train_70_scaled_df,
        'X_val_scaled': x_val_15_scaled_df,
        'X_test_scaled': x_test_15_scaled_df,
        'y_train': y_train_70,
        'y_val': y_val_15,
        'y_test': y_test_15,
        'scaler': scaler_70_15_15
    },
    '60_20_20': {
        'X_train_scaled': x_train_60_scaled_df,
        'X_val_scaled': x_val_20_scaled_df,
        'X_test_scaled': x_test_20_scaled_df,
        'y_train': y_train_60,
        'y_val': y_val_20,
        'y_test': y_test_20,
        'scaler': scaler_60_20_20
    }
}

# Display the first five rows of the scaled training features for the default 70/15/15 split
x_train_70_scaled_df.head()

In [ ]:
# Store default scaled datasets for easier use in the next preprocessing steps
X_train_scaled_df = x_train_70_scaled_df # Default scaled training features
X_val_scaled_df = x_val_15_scaled_df # Default scaled validation features
X_test_scaled_df = x_test_15_scaled_df # Default scaled test features

# Prepare Datasets for Logistic Regression and SVM
Logistic Regression and SVM are sensitive to feature scale, so the scaled datasets are prepared for both algorithms. The same train, validation and test splits are reused to ensure a fair comparison.

In [ ]:
# Prepare explicit scaled datasets for Logistic Regression using 80/10/10 split
lr_x_train_80 = x_train_80_scaled_df # Logistic Regression training features for 80/10/10 split
lr_x_val_10 = x_val_10_scaled_df # Logistic Regression validation features for 80/10/10 split
lr_x_test_10 = x_test_10_scaled_df # Logistic Regression test features for 80/10/10 split
lr_y_train_80 = y_train_80 # Logistic Regression training target for 80/10/10 split
lr_y_val_10 = y_val_10 # Logistic Regression validation target for 80/10/10 split
lr_y_test_10 = y_test_10 # Logistic Regression test target for 80/10/10 split

# Prepare explicit scaled datasets for Logistic Regression using 70/15/15 split
lr_x_train_70 = x_train_70_scaled_df # Logistic Regression training features for 70/15/15 split
lr_x_val_15 = x_val_15_scaled_df # Logistic Regression validation features for 70/15/15 split
lr_x_test_15 = x_test_15_scaled_df # Logistic Regression test features for 70/15/15 split
lr_y_train_70 = y_train_70 # Logistic Regression training target for 70/15/15 split
lr_y_val_15 = y_val_15 # Logistic Regression validation target for 70/15/15 split
lr_y_test_15 = y_test_15 # Logistic Regression test target for 70/15/15 split

# Prepare explicit scaled datasets for Logistic Regression using 60/20/20 split
lr_x_train_60 = x_train_60_scaled_df # Logistic Regression training features for 60/20/20 split
lr_x_val_20 = x_val_20_scaled_df # Logistic Regression validation features for 60/20/20 split
lr_x_test_20 = x_test_20_scaled_df # Logistic Regression test features for 60/20/20 split
lr_y_train_60 = y_train_60 # Logistic Regression training target for 60/20/20 split
lr_y_val_20 = y_val_20 # Logistic Regression validation target for 60/20/20 split
lr_y_test_20 = y_test_20 # Logistic Regression test target for 60/20/20 split

# Prepare explicit scaled datasets for SVM using 80/10/10 split
svm_x_train_80 = x_train_80_scaled_df # SVM training features for 80/10/10 split
svm_x_val_10 = x_val_10_scaled_df # SVM validation features for 80/10/10 split
svm_x_test_10 = x_test_10_scaled_df # SVM test features for 80/10/10 split
svm_y_train_80 = y_train_80 # SVM training target for 80/10/10 split
svm_y_val_10 = y_val_10 # SVM validation target for 80/10/10 split
svm_y_test_10 = y_test_10 # SVM test target for 80/10/10 split

# Prepare explicit scaled datasets for SVM using 70/15/15 split
svm_x_train_70 = x_train_70_scaled_df # SVM training features for 70/15/15 split
svm_x_val_15 = x_val_15_scaled_df # SVM validation features for 70/15/15 split
svm_x_test_15 = x_test_15_scaled_df # SVM test features for 70/15/15 split
svm_y_train_70 = y_train_70 # SVM training target for 70/15/15 split
svm_y_val_15 = y_val_15 # SVM validation target for 70/15/15 split
svm_y_test_15 = y_test_15 # SVM test target for 70/15/15 split

# Prepare explicit scaled datasets for SVM using 60/20/20 split
svm_x_train_60 = x_train_60_scaled_df # SVM training features for 60/20/20 split
svm_x_val_20 = x_val_20_scaled_df # SVM validation features for 60/20/20 split
svm_x_test_20 = x_test_20_scaled_df # SVM test features for 60/20/20 split
svm_y_train_60 = y_train_60 # SVM training target for 60/20/20 split
svm_y_val_20 = y_val_20 # SVM validation target for 60/20/20 split
svm_y_test_20 = y_test_20 # SVM test target for 60/20/20 split

# Create dictionaries to store model-ready datasets for Logistic Regression and SVM
logistic_regression_datasets = {
    '80_10_10': {'X_train': lr_x_train_80, 'X_val': lr_x_val_10, 'X_test': lr_x_test_10, 'y_train': lr_y_train_80, 'y_val': lr_y_val_10, 'y_test': lr_y_test_10},
    '70_15_15': {'X_train': lr_x_train_70, 'X_val': lr_x_val_15, 'X_test': lr_x_test_15, 'y_train': lr_y_train_70, 'y_val': lr_y_val_15, 'y_test': lr_y_test_15},
    '60_20_20': {'X_train': lr_x_train_60, 'X_val': lr_x_val_20, 'X_test': lr_x_test_20, 'y_train': lr_y_train_60, 'y_val': lr_y_val_20, 'y_test': lr_y_test_20}
}

svm_datasets = {
    '80_10_10': {'X_train': svm_x_train_80, 'X_val': svm_x_val_10, 'X_test': svm_x_test_10, 'y_train': svm_y_train_80, 'y_val': svm_y_val_10, 'y_test': svm_y_test_10},
    '70_15_15': {'X_train': svm_x_train_70, 'X_val': svm_x_val_15, 'X_test': svm_x_test_15, 'y_train': svm_y_train_70, 'y_val': svm_y_val_15, 'y_test': svm_y_test_15},
    '60_20_20': {'X_train': svm_x_train_60, 'X_val': svm_x_val_20, 'X_test': svm_x_test_20, 'y_train': svm_y_train_60, 'y_val': svm_y_val_20, 'y_test': svm_y_test_20}
}

print('Prepared explicit Logistic Regression datasets for 80/10/10, 70/15/15 and 60/20/20 splits.') # Display Logistic Regression preparation status
print('Prepared explicit SVM datasets for 80/10/10, 70/15/15 and 60/20/20 splits.') # Display SVM preparation status

In [ ]:
# Initialize model objects for later model training
logistic_regression_model = LogisticRegression(max_iter=1000, random_state=42) # Create Logistic Regression model
svm_model = SVC(kernel='rbf', random_state=42) # Create SVM model with RBF kernel

print(logistic_regression_model) # Display Logistic Regression model configuration
print(svm_model) # Display SVM model configuration

# Feature Selection
Perform feature selection to select the relevant features.
______________________________________________________________________________________
Description: Forward Sequential Feature Selection is applied separately for Logistic Regression and SVM with an RBF kernel across all three split ratios. A smaller set of candidate feature counts is tested first to reduce runtime while still showing the relationship between feature count and 5-fold cross-validation accuracy.

In [ ]:
# Create a stratified 5-fold cross-validation object to preserve class proportions in each fold
sfs_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Use fewer candidate k values first to reduce SFS runtime
candidate_feature_counts = [5, 10, 15, 20, 25, 30, X_train_scaled_df.shape[1]]

# Remove duplicate candidate values and keep only valid feature counts
feature_counts = sorted(set(k for k in candidate_feature_counts if 1 <= k <= X_train_scaled_df.shape[1]))

# Define the two models used for Sequential Feature Selection
sfs_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42), # Logistic Regression model
    'SVM RBF': SVC(kernel='rbf', random_state=42) # SVM model using the RBF kernel
}

# Store all scaled training datasets for all three split ratios
sfs_input_data = {
    '80_10_10': {
        'Logistic Regression': {'X_train': lr_x_train_80, 'y_train': lr_y_train_80},
        'SVM RBF': {'X_train': svm_x_train_80, 'y_train': svm_y_train_80}
    },
    '70_15_15': {
        'Logistic Regression': {'X_train': lr_x_train_70, 'y_train': lr_y_train_70},
        'SVM RBF': {'X_train': svm_x_train_70, 'y_train': svm_y_train_70}
    },
    '60_20_20': {
        'Logistic Regression': {'X_train': lr_x_train_60, 'y_train': lr_y_train_60},
        'SVM RBF': {'X_train': svm_x_train_60, 'y_train': svm_y_train_60}
    }
}

# Create empty containers to store detailed SFS results and summary rows
sfs_results = {}
sfs_summary_rows = []

# Loop through each split ratio so feature selection can be compared across training sizes
for split_name, split_data in sfs_input_data.items():
    sfs_results[split_name] = {}

    # Loop through each model so Logistic Regression and SVM are evaluated separately
    for model_name, model in sfs_models.items():
        X_sfs = split_data[model_name]['X_train'] # Select the scaled training features for the current split and model
        y_sfs = split_data[model_name]['y_train'] # Select the training target values for the current split and model
        model_results = [] # Store all candidate-k results for the current split and model

        # Test each candidate number of selected features
        for k in feature_counts:
            # If k equals the total number of features, use all features without running SFS
            if k == X_sfs.shape[1]:
                selected_indices = np.arange(X_sfs.shape[1]) # Store all feature indices
                selected_features = X_sfs.columns.tolist() # Store all feature names
                selected_X = X_sfs # Use the full feature set
            else:
                # Initialize Forward Sequential Feature Selection for the current number of features
                sfs = SequentialFeatureSelector(
                    estimator=model,
                    n_features_to_select=k,
                    direction='forward',
                    scoring='accuracy',
                    cv=sfs_cv,
                    n_jobs=-1
                )

                # Fit SFS on the current training split to identify the best k features
                sfs.fit(X_sfs, y_sfs)

                # Get the selected feature mask, indices, names and feature subset
                selected_mask = sfs.get_support()
                selected_indices = np.where(selected_mask)[0]
                selected_features = X_sfs.columns[selected_mask].tolist()
                selected_X = X_sfs.iloc[:, selected_indices]

            # Evaluate the selected feature subset using 5-fold cross-validation accuracy
            cv_scores = cross_val_score(
                model,
                selected_X,
                y_sfs,
                scoring='accuracy',
                cv=sfs_cv,
                n_jobs=-1
            )

            # Save the candidate-k result for the current split and model
            model_results.append({
                'split_ratio': split_name,
                'model': model_name,
                'number_of_features': k,
                'cv_accuracy': cv_scores.mean(),
                'cv_std': cv_scores.std(),
                'selected_indices': selected_indices.tolist(),
                'selected_features': selected_features
            })

            # Print progress so the long-running SFS process can be monitored
            print(f'{split_name} | {model_name} | k={k} | CV accuracy={cv_scores.mean():.4f}')

        # Convert the detailed results into a DataFrame for easier viewing and plotting
        result_df = pd.DataFrame(model_results)
        sfs_results[split_name][model_name] = result_df

        # Store the best candidate-k row for the current split and model
        best_row = result_df.loc[result_df['cv_accuracy'].idxmax()]
        sfs_summary_rows.append(best_row)

# Combine the best rows into one summary table
sfs_summary_df = pd.DataFrame(sfs_summary_rows).reset_index(drop=True)

# Display the summary table for all split and model combinations
sfs_summary_df[['split_ratio', 'model', 'number_of_features', 'cv_accuracy', 'cv_std', 'selected_indices', 'selected_features']]

In [ ]:
# Create subplots to compare feature count against cross-validation accuracy for each split ratio
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

# Plot one chart per split ratio
for ax, split_name in zip(axes, sfs_results.keys()):
    # Plot one accuracy curve for each model within the current split ratio
    for model_name, result_df in sfs_results[split_name].items():
        ax.plot(
            result_df['number_of_features'],
            result_df['cv_accuracy'],
            marker='o',
            label=model_name
        )

    # Add title, labels, ticks and grid for the current subplot
    ax.set_title(f'{split_name} Split')
    ax.set_xlabel('Number of Features')
    ax.set_xticks(feature_counts)
    ax.grid(True, linestyle='--', alpha=0.6)

# Add a shared y-axis label and legend
axes[0].set_ylabel('Cross-Validation Accuracy')
axes[-1].legend()

# Add an overall title for all three split-ratio plots
fig.suptitle('Sequential Feature Selection Across Split Ratios')
plt.tight_layout()
plt.show()

In [ ]:
# Select the overall best Logistic Regression row across all split ratios
lr_optimal_row = sfs_summary_df[sfs_summary_df['model'] == 'Logistic Regression'].sort_values(
    by='cv_accuracy',
    ascending=False
).iloc[0]

# Select the overall best SVM RBF row across all split ratios
svm_optimal_row = sfs_summary_df[sfs_summary_df['model'] == 'SVM RBF'].sort_values(
    by='cv_accuracy',
    ascending=False
).iloc[0]

# Extract the best split ratio and optimal number of features for Logistic Regression
lr_optimal_split = lr_optimal_row['split_ratio']
lr_optimal_k = int(lr_optimal_row['number_of_features'])

# Extract the best split ratio and optimal number of features for SVM RBF
svm_optimal_split = svm_optimal_row['split_ratio']
svm_optimal_k = int(svm_optimal_row['number_of_features'])

# Extract selected feature indices for each best model result
lr_selected_feature_indices = lr_optimal_row['selected_indices']
svm_selected_feature_indices = svm_optimal_row['selected_indices']

# Extract selected feature names for easier interpretation
lr_selected_feature_names = lr_optimal_row['selected_features']
svm_selected_feature_names = svm_optimal_row['selected_features']

# Display the best SFS result for Logistic Regression across all split ratios
print('Overall Best Logistic Regression SFS Results')
print('Best split ratio:', lr_optimal_split)
print('Optimal number of features:', lr_optimal_k)
print('Selected feature indices:', lr_selected_feature_indices)
print('Selected feature names:', lr_selected_feature_names)
print('Best 5-fold CV accuracy:', round(lr_optimal_row['cv_accuracy'], 4))
print('-' * 60)

# Display the best SFS result for SVM with RBF kernel across all split ratios
print('Overall Best SVM RBF SFS Results')
print('Best split ratio:', svm_optimal_split)
print('Optimal number of features:', svm_optimal_k)
print('Selected feature indices:', svm_selected_feature_indices)
print('Selected feature names:', svm_selected_feature_names)
print('Best 5-fold CV accuracy:', round(svm_optimal_row['cv_accuracy'], 4))

# Create a dictionary that maps each split ratio to its corresponding Logistic Regression datasets
lr_split_lookup = {
    '80_10_10': {'X_train': lr_x_train_80, 'X_val': lr_x_val_10, 'X_test': lr_x_test_10, 'y_train': lr_y_train_80, 'y_val': lr_y_val_10, 'y_test': lr_y_test_10},
    '70_15_15': {'X_train': lr_x_train_70, 'X_val': lr_x_val_15, 'X_test': lr_x_test_15, 'y_train': lr_y_train_70, 'y_val': lr_y_val_15, 'y_test': lr_y_test_15},
    '60_20_20': {'X_train': lr_x_train_60, 'X_val': lr_x_val_20, 'X_test': lr_x_test_20, 'y_train': lr_y_train_60, 'y_val': lr_y_val_20, 'y_test': lr_y_test_20}
}

# Create a dictionary that maps each split ratio to its corresponding SVM RBF datasets
svm_split_lookup = {
    '80_10_10': {'X_train': svm_x_train_80, 'X_val': svm_x_val_10, 'X_test': svm_x_test_10, 'y_train': svm_y_train_80, 'y_val': svm_y_val_10, 'y_test': svm_y_test_10},
    '70_15_15': {'X_train': svm_x_train_70, 'X_val': svm_x_val_15, 'X_test': svm_x_test_15, 'y_train': svm_y_train_70, 'y_val': svm_y_val_15, 'y_test': svm_y_test_15},
    '60_20_20': {'X_train': svm_x_train_60, 'X_val': svm_x_val_20, 'X_test': svm_x_test_20, 'y_train': svm_y_train_60, 'y_val': svm_y_val_20, 'y_test': svm_y_test_20}
}

# Select the best split-ratio dataset for Logistic Regression and keep only the selected SFS features
lr_best_split_data = lr_split_lookup[lr_optimal_split]
lr_x_train_sfs = lr_best_split_data['X_train'].iloc[:, lr_selected_feature_indices]
lr_x_val_sfs = lr_best_split_data['X_val'].iloc[:, lr_selected_feature_indices]
lr_x_test_sfs = lr_best_split_data['X_test'].iloc[:, lr_selected_feature_indices]

# Select the best split-ratio dataset for SVM RBF and keep only the selected SFS features
svm_best_split_data = svm_split_lookup[svm_optimal_split]
svm_x_train_sfs = svm_best_split_data['X_train'].iloc[:, svm_selected_feature_indices]
svm_x_val_sfs = svm_best_split_data['X_val'].iloc[:, svm_selected_feature_indices]
svm_x_test_sfs = svm_best_split_data['X_test'].iloc[:, svm_selected_feature_indices]

# Store the Logistic Regression SFS-ready datasets and selected feature information
logistic_regression_sfs_dataset = {
    'split_ratio': lr_optimal_split,
    'X_train': lr_x_train_sfs,
    'X_val': lr_x_val_sfs,
    'X_test': lr_x_test_sfs,
    'y_train': lr_best_split_data['y_train'],
    'y_val': lr_best_split_data['y_val'],
    'y_test': lr_best_split_data['y_test'],
    'selected_indices': lr_selected_feature_indices,
    'selected_features': lr_selected_feature_names
}

# Store the SVM RBF SFS-ready datasets and selected feature information
svm_sfs_dataset = {
    'split_ratio': svm_optimal_split,
    'X_train': svm_x_train_sfs,
    'X_val': svm_x_val_sfs,
    'X_test': svm_x_test_sfs,
    'y_train': svm_best_split_data['y_train'],
    'y_val': svm_best_split_data['y_val'],
    'y_test': svm_best_split_data['y_test'],
    'selected_indices': svm_selected_feature_indices,
    'selected_features': svm_selected_feature_names
}

# Display the final shapes after applying SFS-selected features
print('Logistic Regression SFS dataset split:', lr_optimal_split)
print('Logistic Regression SFS dataset shapes:')
print(lr_x_train_sfs.shape, lr_x_val_sfs.shape, lr_x_test_sfs.shape)
print('SVM RBF SFS dataset split:', svm_optimal_split)
print('SVM RBF SFS dataset shapes:')
print(svm_x_train_sfs.shape, svm_x_val_sfs.shape, svm_x_test_sfs.shape)

# Preprocessing Summary
- The CSV file was loaded without headers using column names from the dermatology README.
- The `?` missing marker was converted into `NaN`.
- Missing `age` values were imputed using the median age.
- Duplicate rows were checked.
- IQR-based row removal was not applied because most features are valid ordinal medical scores from 0 to 3.
- The final class distribution table and plot were placed after outlier checking.
- Stratified 80/10/10, 70/15/15, and 60/20/20 train/validation/test splits were prepared.
- Standardization was fitted on the training set only for each split ratio and applied to validation and test sets.
- Scaled datasets were prepared for Logistic Regression and SVM.
- Forward Sequential Feature Selection was applied for Logistic Regression and SVM with an RBF kernel across all three split ratios.
- Candidate feature counts `[5, 10, 15, 20, 25, 30, 34]` were evaluated first using 5-fold cross-validation accuracy to reduce runtime.
- The best split ratio, optimal number of features, selected feature indices, selected feature names, and model-specific SFS datasets were prepared.